# Immune integration: RNA-only training

Train `AmbientRegularizedSCVI` on 7-dataset immune integration (bone marrow + PBMC).
Uses early stopping with stratified validation split.

**Input**: `results/immune_integration/adata_rna.h5ad` + `scrublet_results.csv`
**Output**: Trained model + latent representation + UMAP

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import scipy
import torch
import os

import scvi
import regularizedvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

## Data loading

In [ ]:
from data_loading_utils import load_atac_qc_metrics, load_scrublet_scores

data_folder = "results/immune_integration/"
rna_path = os.path.join(data_folder, "adata_rna.h5ad")
scrublet_path = os.path.join(data_folder, "scrublet_results.csv")
atac_qc_path = os.path.join(data_folder, "atac_qc_metrics.csv")

adata = sc.read_h5ad(rna_path)
print(f"Loaded: {adata.shape}")

# Load scrublet and ATAC QC using safe reindex utilities
adata = load_scrublet_scores(adata, scrublet_path)
adata = load_atac_qc_metrics(adata, atac_qc_path)

# Ensure counts in X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

print(f"\nDatasets: {adata.obs['dataset'].nunique()}")
print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"Donors: {adata.obs['donor'].nunique()}")

## QC summary

In [ ]:
# Per-dataset cell counts and QC metrics
for ds in sorted(adata.obs["dataset"].unique()):
    mask = adata.obs["dataset"] == ds
    sub = adata.obs.loc[mask]
    n_cells = mask.sum()
    n_batches = sub["batch"].nunique()
    print(f"\n{ds}: {n_cells:,} cells, {n_batches} batches")

    # Count-based QC
    total_counts = np.array(adata[mask].X.sum(1)).squeeze()
    n_genes = np.array((adata[mask].X > 0).sum(1)).squeeze()
    print(f"  Counts: median={np.median(total_counts):.0f}, range=[{total_counts.min():.0f}, {total_counts.max():.0f}]")
    print(f"  Genes:  median={np.median(n_genes):.0f}, range=[{n_genes.min():.0f}, {n_genes.max():.0f}]")

    if "mt_frac" in sub.columns:
        print(f"  MT frac: median={sub['mt_frac'].median():.3f}")
    if "doublet_score" in sub.columns:
        n_doublet = (sub["doublet_score"] > 0.18).sum()
        print(f"  Doublets (>0.18): {n_doublet} ({100 * n_doublet / n_cells:.1f}%)")

    # Annotations
    if "harmonized_annotation" in sub.columns:
        n_annot = sub["harmonized_annotation"].notna().sum()
        print(f"  Annotated: {n_annot}/{n_cells} ({100 * n_annot / n_cells:.1f}%)")

In [ ]:
rcParams["figure.figsize"] = 12, 3
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

total_counts = np.array(adata.X.sum(1)).squeeze()
n_genes = np.array((adata.X > 0).sum(1)).squeeze()

axes[0].hist(np.log10(total_counts[total_counts > 0]), bins=100)
axes[0].axvline(np.log10(1000), color="red", linestyle="--")
axes[0].axvline(np.log10(80000), color="red", linestyle="--")
axes[0].set_xlabel("log10(UMI counts)")
axes[0].set_title(f"Total counts (n={len(total_counts)}, zero={np.sum(total_counts == 0)})")

axes[1].hist(np.log10(n_genes[n_genes > 0]), bins=100)
axes[1].axvline(np.log10(500), color="red", linestyle="--")
axes[1].axvline(np.log10(10000), color="red", linestyle="--")
axes[1].set_xlabel("log10(genes)")
axes[1].set_title("Genes detected")

if "mt_frac" in adata.obs.columns:
    axes[2].hist(adata.obs["mt_frac"].dropna(), bins=100)
    axes[2].axvline(0.20, color="red", linestyle="--")
    axes[2].set_xlabel("MT fraction")
    axes[2].set_title("Mitochondrial fraction")

if "doublet_score" in adata.obs.columns:
    axes[3].hist(adata.obs["doublet_score"].dropna(), bins=100)
    axes[3].axvline(0.18, color="red", linestyle="--")
    axes[3].set_xlabel("Doublet score")
    axes[3].set_title("Scrublet doublet score")

plt.tight_layout()
plt.show()

## Cell filtering

In [ ]:
from data_loading_utils import adaptive_qc_filter

adata = adaptive_qc_filter(
    adata,
    min_counts=900,
    min_genes=600,
    min_fragments=1500,
    per_dataset_quantile=0.20,
)

## Gene filtering

In [ ]:
from regularizedvi.utils import filter_genes

selected = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.04,
)
adata = adata[:, selected].copy()
print(f"Selected {adata.n_vars:,} genes")

In [ ]:
adata.layers["counts"] = adata.X

## Model parameters

In [ ]:
# Papermill parameters
results_folder = "results/immune_integration_rna/"
additive_bg_prior_alpha = 1.0
additive_bg_prior_beta = 100.0
use_additive_background = 1
regularise_background = 0
compute_pearson = 1
use_feature_scaling = 1
library_log_means_centering_sensitivity = 1.0
library_log_vars_weight = 0.5
px_r_init_mean = None
px_r_init_std = None
additive_bg_init_mean = None
additive_bg_init_std = None
stratify_validation_key = "harmonized_annotation+batch"
early_stopping_min_delta_per_feature = 0.0002
early_stopping_patience = 10
max_epochs = 2000
wandb_project = "regularizedVI"
wandb_name = "immune_rna_base"
wandb_entity = None
wandb_notes = "RNA: libvar=0.5, stratified, ES=2e-4, patience=10. 7-dataset immune integration, 512h/128z."
wandb_group = "immune_integration"

In [ ]:
from regularizedvi.utils import coerce_papermill_params, finish_wandb, log_figure_to_wandb, setup_wandb_logger

params = coerce_papermill_params(
    additive_bg_prior_alpha=(additive_bg_prior_alpha, float),
    additive_bg_prior_beta=(additive_bg_prior_beta, float),
    use_additive_background=(use_additive_background, bool),
    regularise_background=(regularise_background, bool),
    compute_pearson=(compute_pearson, bool),
    use_feature_scaling=(use_feature_scaling, bool),
    library_log_means_centering_sensitivity=(library_log_means_centering_sensitivity, "float_or_none"),
    library_log_vars_weight=(library_log_vars_weight, float),
    px_r_init_mean=(px_r_init_mean, "float_or_none"),
    px_r_init_std=(px_r_init_std, "float_or_none"),
    additive_bg_init_mean=(additive_bg_init_mean, "float_or_none"),
    additive_bg_init_std=(additive_bg_init_std, "float_or_none"),
    stratify_validation_key=(stratify_validation_key, "str_or_none"),
    early_stopping_min_delta_per_feature=(early_stopping_min_delta_per_feature, "float_or_none"),
    early_stopping_patience=(early_stopping_patience, int),
    max_epochs=(max_epochs, int),
    wandb_project=(wandb_project, "str_or_none"),
    wandb_name=(wandb_name, "str_or_none"),
    wandb_entity=(wandb_entity, "str_or_none"),
    wandb_notes=(wandb_notes, "str_or_none"),
    wandb_group=(wandb_group, "str_or_none"),
)
additive_bg_prior_alpha = params["additive_bg_prior_alpha"]
additive_bg_prior_beta = params["additive_bg_prior_beta"]
use_additive_background = params["use_additive_background"]
regularise_background = params["regularise_background"]
compute_pearson = params["compute_pearson"]
use_feature_scaling = params["use_feature_scaling"]
library_log_means_centering_sensitivity = params["library_log_means_centering_sensitivity"]
library_log_vars_weight = params["library_log_vars_weight"]
px_r_init_mean = params["px_r_init_mean"]
px_r_init_std = params["px_r_init_std"]
additive_bg_init_mean = params["additive_bg_init_mean"]
additive_bg_init_std = params["additive_bg_init_std"]
stratify_validation_key = params["stratify_validation_key"]
early_stopping_min_delta_per_feature = params["early_stopping_min_delta_per_feature"]
early_stopping_patience = params["early_stopping_patience"]
max_epochs = params["max_epochs"]
wandb_project = params["wandb_project"]
wandb_name = params["wandb_name"]
wandb_entity = params["wandb_entity"]
wandb_notes = params["wandb_notes"]
wandb_group = params["wandb_group"]

os.makedirs(results_folder, exist_ok=True)

wandb_loggers, wandb_run = setup_wandb_logger(
    wandb_project=wandb_project,
    wandb_name=wandb_name,
    wandb_entity=wandb_entity,
    wandb_notes=wandb_notes,
    wandb_group=wandb_group,
    config={
        "additive_bg_prior_alpha": additive_bg_prior_alpha,
        "additive_bg_prior_beta": additive_bg_prior_beta,
        "use_additive_background": use_additive_background,
        "regularise_background": regularise_background,
        "compute_pearson": compute_pearson,
        "use_feature_scaling": use_feature_scaling,
        "library_log_means_centering_sensitivity": library_log_means_centering_sensitivity,
        "library_log_vars_weight": library_log_vars_weight,
        "stratify_validation_key": stratify_validation_key,
        "early_stopping_min_delta_per_feature": early_stopping_min_delta_per_feature,
        "early_stopping_patience": early_stopping_patience,
        "n_hidden": 512,
        "n_latent": 128,
        "n_layers": 1,
    },
    results_folder=results_folder,
)

## Model setup and training

In [ ]:
regularizedvi.AmbientRegularizedSCVI.setup_anndata(
    adata,
    layer="counts",
    ambient_covariate_keys=["batch"],
    nn_conditioning_covariate_keys=["dataset", "donor"],
    feature_scaling_covariate_keys=["dataset", "donor"] if use_feature_scaling else [],
    dispersion_key="batch",
    library_size_key="batch",
)

In [ ]:
model = regularizedvi.AmbientRegularizedSCVI(
    adata,
    n_hidden=512,
    n_layers=1,
    n_latent=128,
    use_additive_background=use_additive_background,
    additive_bg_prior_alpha=additive_bg_prior_alpha,
    additive_bg_prior_beta=additive_bg_prior_beta,
    regularise_background=regularise_background,
    compute_pearson=compute_pearson,
    library_log_means_centering_sensitivity=library_log_means_centering_sensitivity,
    library_log_vars_weight=library_log_vars_weight,
    px_r_init_mean=px_r_init_mean,
    px_r_init_std=px_r_init_std,
    additive_bg_init_mean=additive_bg_init_mean,
    additive_bg_init_std=additive_bg_init_std,
)

print(f"Additive background: {use_additive_background}")
print(f"Regularise background: {regularise_background}")
print(f"Compute Pearson: {compute_pearson}")
print(f"Library centering sensitivity: {library_log_means_centering_sensitivity}")
print(f"Library log vars weight: {library_log_vars_weight}")

In [ ]:
import time
from scvi.train import SaveCheckpoint

checkpoint_cb = SaveCheckpoint(
    dirpath=f"{results_folder}/checkpoints",
    every_n_epochs=200,
    save_top_k=-1,
    filename="epoch-{epoch}",
)

_es_kwargs = {}
if early_stopping_min_delta_per_feature is not None:
    _es_kwargs["early_stopping_min_delta_per_feature"] = early_stopping_min_delta_per_feature

_ds_kwargs = {"num_workers": 7}
if stratify_validation_key is not None:
    from sklearn.model_selection import train_test_split

    keys = stratify_validation_key.split("+")
    strat_labels = adata.obs[keys[0]].astype(str)
    for k in keys[1:]:
        strat_labels = strat_labels + "_" + adata.obs[k].astype(str)
    counts = strat_labels.value_counts()
    rare = counts[counts < 2].index
    if len(rare) > 0:
        strat_labels = strat_labels.replace(rare, "_rare_")
        print(f"Merged {len(rare)} rare strata into '_rare_'")
    all_idx = np.arange(adata.n_obs)
    train_idx, val_idx = train_test_split(all_idx, test_size=0.1, stratify=strat_labels.values, random_state=42)
    _ds_kwargs["external_indexing"] = [train_idx, val_idx]
    print(f"Stratified split by {stratify_validation_key}: {len(train_idx)} train, {len(val_idx)} val")

t0 = time.time()
model.train(
    check_val_every_n_epoch=1,
    train_size=0.9,
    max_epochs=max_epochs,
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=early_stopping_patience,
    early_stopping_monitor="elbo_validation",
    enable_checkpointing=True,
    callbacks=[checkpoint_cb],
    logger=wandb_loggers,
    datasplitter_kwargs=_ds_kwargs,
    **_es_kwargs,
)
elapsed = time.time() - t0
n_epochs = len(model.history_["elbo_train"])
print(f"Training: {elapsed / 60:.1f} min, {n_epochs} epochs, {elapsed / n_epochs:.2f} s/epoch")

## Results

In [ ]:
fig = model.plot_training_diagnostics(skip_epochs=80)
log_figure_to_wandb("training_diagnostics", fig)
plt.show()

In [ ]:
ref_run_name = f"{results_folder}/model"
model.save(ref_run_name, overwrite=True)

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
print(f"Latent representation shape: {latent.shape}")

In [ ]:
k = 50
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=k, metric="euclidean")
sc.tl.umap(adata, min_dist=0.4, spread=1.3)
sc.tl.leiden(adata, resolution=12, flavor="igraph")

In [ ]:
color = ["dataset", "batch", "harmonized_annotation", "level_1"]

rcParams["figure.figsize"] = 7, 7
sc.pl.umap(
    adata,
    color=color,
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=8,
)

In [ ]:
# Leiden clusters
rcParams["figure.figsize"] = 7, 7
sc.pl.umap(
    adata,
    color="leiden",
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    legend_fontsize=8,
)

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
os.makedirs(output_dir, exist_ok=True)

X_scVI = pd.DataFrame(
    adata.obsm["X_scVI"],
    index=adata.obs_names,
    columns=range(adata.obsm["X_scVI"].shape[1]),
)
X_scVI.to_csv(f"{output_dir}/X_scVI.csv")

X_umap = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,
    columns=range(2),
)
X_umap.to_csv(f"{output_dir}/X_umap_k{k}.csv")

adata.obs[["leiden"]].to_csv(f"{output_dir}/leiden_k{k}.csv")

scipy.sparse.save_npz(f"{output_dir}/distances_euclidean_k{k}.npz", adata.obsp["distances"], compressed=True)
scipy.sparse.save_npz(f"{output_dir}/connectivities_euclidean_k{k}.npz", adata.obsp["connectivities"], compressed=True)

print(f"Outputs saved to {output_dir}")

In [ ]:
finish_wandb()